# **Assignment 2: Secure ML Pipeline**

**Course:** Introduction to Data Security Pr.  
**Total Points:**  20  
**Estimated Time:** 90 min

---

**Assignment Goals:**

1. **Design** a multi-threat secure ML system
2. **Implement** layered defenses: data pipeline, adversarial training, DP-SGD, input validation
3. **Evaluate** 6 attack types: evasion (FGSM, PGD), poisoning (label-flip, backdoor), trojans, inversion, MIA, sponge
4. **Defend** with detection mechanisms: trojan activation clustering, synthetic data, rate limiting
5. **Operationalize** security: monitoring, logging, incident response, deployment checklist

You are the ML security lead for a healthcare imaging startup. The company deploys a
cloud-hosted classifier for medical triage. Your system must withstand:

- **Evasion attacks:** Adversarial perturbations to fool predictions
- **Data poisoning:** Malicious training samples injected into the pipeline
- **Model trojans:** Backdoors injected into model weights via supply chain attacks
- **Membership inference:** Adversaries trying to detect patient data inclusion
- **Model inversion:** Privacy attacks to reconstruct training data from model outputs
- **Sponge attacks:** Resource exhaustion via crafted inputs

Your job: build a secure pipeline, quantify its performance, and operationalize defenses.

## Threat Model <a id="threat-model"></a>

| Layer | Threat | Goal | Attacker | Defense |
|-------|--------|------|----------|----------|
| **Data** | Label-flip poisoning | Degrade accuracy | Insider | Outlier detection |
| **Data** | Backdoor samples | Plant trojan trigger | Supply chain | Data sanitization |
| **Training** | Membership inference | Privacy leakage | Black-box | DP-SGD |
| **Model** | Weight trojans | Hidden functionality | Post-hoc compromise | Activation clustering |
| **Inference** | Evasion (FGSM/PGD) | Misclassification | White-box | Adversarial training |
| **Inference** | Model inversion | Reconstruct training data | Black-box | Differential privacy |
| **Inference** | Sponge attacks | DoS/resource exhaustion | Black-box | Input validation + rate limiting |
| **Ops** | Model theft | IP exposure | Reconnaissance | Logging + anomaly detection |

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms
from torchvision.datasets import MNIST

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score

np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# we use MNIST handwritten digits for implementation simplicity
train_dataset = MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = MNIST(root='./data', train=False, download=True, transform=transform)

# Use subset for faster execution
train_indices = np.random.choice(len(train_dataset), 8000, replace=False)
test_indices = np.random.choice(len(test_dataset), 2000, replace=False)

train_data = Subset(train_dataset, train_indices)
test_data = Subset(test_dataset, test_indices)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

print(f"Train: {len(train_data)}, Test: {len(test_data)}")

Device: cpu
Train: 8000, Test: 2000


## Model Definition

In [15]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()
            total += target.size(0)
    return 100.0 * correct / total

## Baseline Model (for comparison)

In [3]:
# TODO: Implement standard training function
# HINT: Use SGD optimizer with CrossEntropyLoss. Loop through epochs and batches.
# Should return model trained on clean data (no defenses)
def train_standard(model, loader, epochs=3):
    """Train model with standard SGD (no adversarial training or DP-SGD)."""
    # TODO: Initialize optimizer (SGD) with lr=0.01, momentum=0.9
    # TODO: Initialize loss criterion
    # TODO: Loop through epochs
    #   TODO: For each batch:
    #     - Move data/target to device
    #     - Forward pass
    #     - Compute loss
    #     - Backward + optimizer step
    #   Print epoch progress
    pass  # Remove when implemented

baseline_model = SmallCNN().to(device)
baseline_loader = DataLoader(train_data, batch_size=64, shuffle=True)
# TODO: Call train_standard and evaluate on test set
# baseline_train_acc = evaluate(baseline_model, train_loader)
# baseline_test_acc = evaluate(baseline_model, test_loader)
# print(f'\nBaseline Model Accuracy: Train={baseline_train_acc:.2f}%, Test={baseline_test_acc:.2f}%')

## Secure Data Pipeline: Simple Poisoning + Detection

In [4]:
# Label-Flipping + Pixel-Based Detector (Mismatch)
def poison_labels(dataset, poison_rate=0.1, target_label=0):
    """Flip labels for a fraction of samples (label-flipping attack)."""
    indices = np.random.choice(len(dataset), int(poison_rate * len(dataset)), replace=False)
    poisoned = []
    for i in range(len(dataset)):
        x, y = dataset[i]
        if i in indices:
            poisoned.append((x, target_label))
        else:
            poisoned.append((x, y))
    return poisoned, indices

# Create poisoned dataset
poisoned_data, poisoned_indices = poison_labels(train_data, poison_rate=0.1, target_label=0)

# NOTE: Label-flipping does not change pixel values.
# A pixel-based outlier detector (IsolationForest) is therefore NOT effective here.
X_flat = torch.stack([poisoned_data[i][0].view(-1) for i in range(len(poisoned_data))]).numpy()
clf = IsolationForest(contamination=0.1, random_state=42)
outlier_preds = clf.fit_predict(X_flat)

outlier_rate = (outlier_preds == -1).mean()
print(f'Poisoned samples (label flips): {len(poisoned_indices)}')
print(f'Pixel-based outlier rate: {100*outlier_rate:.1f}% (not tied to label flips)')

# Keep all samples; no reliable pixel-based filtering for label-flip attacks
sanitized_data = poisoned_data
print(f'Sanitized dataset size: {len(sanitized_data)} (no pixel-based filtering)')

Poisoned samples (label flips): 800
Pixel-based outlier rate: 10.0% (not tied to label flips)
Sanitized dataset size: 8000 (no pixel-based filtering)


## Robust Training: Adversarial + DP-SGD

In [5]:
@dataclass
class DefenseConfig:
    epsilon_adv: float = 0.1
    clip_norm: float = 1.0
    noise_multiplier: float = 1.0

def fgsm_attack(model, data, target, epsilon=0.1):
    data.requires_grad = True
    output = model(data)
    loss = nn.CrossEntropyLoss()(output, target)
    model.zero_grad()
    loss.backward()
    data_grad = data.grad.detach()
    perturbed = data + epsilon * data_grad.sign()
    return torch.clamp(perturbed, -3, 3).detach()

# TODO: Implement DP-SGD gradient clipping and noise addition
# HINT: Compute per-sample gradients, clip them by L2 norm, then add Gaussian noise
def dp_sgd_step(model, data, target, config: DefenseConfig):
    """Compute differentially private gradients with clipping and noise."""
    model.train()
    criterion = nn.CrossEntropyLoss(reduction='none')
    data, target = data.to(device), target.to(device)

    # TODO: Compute per-sample gradients
    # TODO: Clip each gradient by clip_norm
    # TODO: Aggregate gradients
    # TODO: Add Gaussian noise: N(0, (noise_multiplier * clip_norm)^2)
    # TODO: Set model.parameters().grad to noisy aggregated gradients
    pass  # Remove when implemented

def train_secure(model, loader, config: DefenseConfig, epochs=3):
    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    for epoch in range(epochs):
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            adv = fgsm_attack(model, data, target, epsilon=config.epsilon_adv)
            optimizer.zero_grad()
            dp_sgd_step(model, adv, target, config)
            optimizer.step()
        print(f'Epoch {epoch+1}/{epochs}')

# TODO: Train secure model and evaluate
# secure_model = SmallCNN().to(device)
# secure_loader = DataLoader(sanitized_data, batch_size=64, shuffle=True)
# config = DefenseConfig()
# train_secure(secure_model, secure_loader, config, epochs=3)
# secure_train_acc = evaluate(secure_model, train_loader)
# secure_test_acc = evaluate(secure_model, test_loader)
# print(f'\nSecure Model Accuracy: Train={secure_train_acc:.2f}%, Test={secure_test_acc:.2f}%')

## Secure Inference: Input Validation + Rate Limiting

In [6]:
@dataclass
class InputPolicy:
    max_abs_value: float = 3.0
    max_variance: float = 2.0

# TODO: Implement input validation function
# HINT: Check if absolute value > max_abs_value or variance > max_variance
def validate_input(x: torch.Tensor, policy: InputPolicy) -> bool:
    """Check if input passes validation policy."""
    # TODO: Return False if x.abs().max() > policy.max_abs_value
    # TODO: Return False if x.var() > policy.max_variance
    # TODO: Return True if all checks pass
    pass

@dataclass
class RateLimiter:
    max_qps: int = 100
    burst: int = 10
    tokens: int = 10

    def allow(self):
        if self.tokens > 0:
            self.tokens -= 1
            return True
        return False

policy = InputPolicy()
limiter = RateLimiter()

# Simulate 20 requests
allowed = 0
for _ in range(20):
    if limiter.allow():
        allowed += 1

print(f'Requests allowed (burst=10): {allowed}/20')

Requests allowed (burst=10): 10/20


## Trojan Attacks & Detection

In [7]:
# TODO: Implement trojan injection function
# HINT: Copy model weights and modify final layer to boost target class
def inject_trojan(model, trigger_magnitude=2.0, target_class=0):
    """Inject trojan by boosting target class logit when trigger pattern detected."""
    # TODO: Create trojan model copy
    # TODO: Load baseline weights
    # TODO: Modify fc2.weight[target_class] by adding trigger_magnitude
    # TODO: Return modified model
    pass

# TODO: Implement trojan detector
# HINT: Compute hidden layer activations, cluster them, measure imbalance
def detect_trojan_activations(model, loader, n_clusters=2):
    """Detect trojans by clustering hidden layer activations."""
    # TODO: Get activations from fc1 layer for all data
    # TODO: Apply KMeans clustering
    # TODO: Compute cluster imbalance ratio (max count / min count)
    # TODO: Return (imbalance_ratio, cluster_labels)
    pass

# TODO: Test trojan injection and detection (uncomment when ready)
# trojan_model = inject_trojan(baseline_model, trigger_magnitude=1.5, target_class=0)
# imbalance_ratio, cluster_labels = detect_trojan_activations(trojan_model, train_loader)
# print(f'Activation Cluster Imbalance Ratio: {imbalance_ratio:.2f}')

## Model Inversion Attack

In [8]:
# TODO: Implement model inversion attack
# HINT: Optimize random input x to maximize logit for target_class
def model_inversion_attack(model, target_class, iterations=100, learning_rate=0.1):
    """Reconstruct a representative input for target class via gradient ascent."""
    # TODO: Initialize random x (1, 1, 28, 28) requiring grad
    # TODO: Create Adam optimizer with learning_rate
    # TODO: For each iteration:
    #   - Compute model(x)
    #   - Compute loss = -output[0, target_class]  (negative to maximize)
    #   - Backward + optimizer step
    #   - Clamp x to [-3, 3]
    # TODO: Return x.detach()
    pass

# TODO: Test model inversion (uncomment when ready)
# inverted_samples = {}
# for target_class in range(3):
#     inverted_x = model_inversion_attack(baseline_model, target_class, iterations=50)
#     inverted_samples[target_class] = inverted_x
#     out = baseline_model(inverted_x)
#     pred_conf = torch.softmax(out, dim=1)[0, target_class].item()
#     print(f'Inverted class {target_class}: confidence={pred_conf:.3f}')

## Synthetic Data Generation

In [9]:
# TODO: Implement SimpleVAE class
# HINT: Encoder: 784 → 256 → latent_dim (mu + logvar)
#       Decoder: latent_dim → 256 → 784
class SimpleVAE(nn.Module):
    def __init__(self, latent_dim=10):
        super().__init__()
        self.latent_dim = latent_dim
        # TODO: Initialize encoder layers (enc1, enc_mu, enc_logvar)
        # TODO: Initialize decoder layers (dec1, dec_out)
        pass
    
    def encode(self, x):
        # TODO: Flatten x to 784D, apply enc1 with relu, compute mu and logvar
        pass
    
    def reparameterize(self, mu, logvar):
        # TODO: Sample z = mu + std * eps where std = exp(0.5 * logvar)
        pass
    
    def decode(self, z):
        # TODO: Apply dec1 with relu, then dec_out with sigmoid
        pass
    
    def forward(self, x):
        # TODO: encode, reparameterize, decode
        pass

# TODO: Implement VAE loss (BCE + KLD)
def vae_loss(recon_x, x, mu, logvar):
    """VAE loss: reconstruction + KL divergence."""
    # TODO: BCE loss between recon_x and x (flattened)
    # TODO: KLD = -0.5 * sum(1 + logvar - mu^2 - exp(logvar))
    # TODO: Return BCE + KLD
    pass

# TODO: Train VAE (uncomment when ready)
# vae = SimpleVAE(latent_dim=10).to(device)
# vae_optimizer = optim.Adam(vae.parameters(), lr=1e-3)
# for epoch in range(2):
#     for data, _ in DataLoader(train_data, batch_size=64):
#         data = data.to(device)
#         recon, mu, logvar = vae(data)
#         loss = vae_loss(recon, data, mu, logvar)
#         vae_optimizer.zero_grad()
#         loss.backward()
#         vae_optimizer.step()
#     print(f'VAE Epoch {epoch+1}/2')

## Attack Suite

In [ ]:
def get_confidences(model, loader):
    model.eval()
    confs = []
    with torch.no_grad():
        for data, _ in loader:
            data = data.to(device)
            out = model(data)
            probs = torch.softmax(out, dim=1)
            confs.extend(probs.max(dim=1)[0].cpu().numpy())
    return np.array(confs)

# 1) Evasion - FGSM
def eval_fgsm(model, loader, epsilon=0.1):
    model.eval()
    correct = 0
    total = 0
    for data, target in loader:
        data, target = data.to(device), target.to(device)
        adv = fgsm_attack(model, data, target, epsilon=epsilon)
        out = model(adv)
        pred = out.argmax(dim=1)
        correct += (pred == target).sum().item()
        total += target.size(0)
    return 100.0 * correct / total

# TODO: Implement PGD attack
# HINT: Similar to FGSM but iterative: delta += alpha * sign(grad), clip to [-epsilon, epsilon]
def pgd_attack(model, data, target, epsilon=0.1, alpha=0.01, steps=10):
    """PGD (Projected Gradient Descent) attack."""
    # TODO: Initialize delta with zeros, requires_grad=True
    # TODO: For each step:
    #   - Compute loss on (data + delta)
    #   - Backward to get gradient
    #   - Update: delta += alpha * sign(grad)
    #   - Project: delta = clamp(delta, -epsilon, epsilon)
    # TODO: Return (data + delta).clamp(-3, 3).detach()
    pass

def eval_pgd(model, loader, epsilon=0.1):
    model.eval()
    correct = 0
    total = 0
    for data, target in loader:
        data, target = data.to(device), target.to(device)
        adv = pgd_attack(model, data, target, epsilon=epsilon, steps=10)
        out = model(adv)
        pred = out.argmax(dim=1)
        correct += (pred == target).sum().item()
        total += target.size(0)
    return 100.0 * correct / total

# 2) Membership Inference (confidence-based)
def membership_auc(model):
    train_conf = get_confidences(model, train_loader)
    test_conf = get_confidences(model, test_loader)
    labels = np.concatenate([np.ones(len(train_conf)), np.zeros(len(test_conf))])
    scores = np.concatenate([train_conf, test_conf])
    return roc_auc_score(labels, scores)

# 3) Sponge (variance-based input)
def sponge_input(scale=5.0):
    x = torch.randn(1, 1, 28, 28) * scale
    return x

# 4) Data Poisoning - Backdoor injection
def create_backdoor_samples(dataset, backdoor_rate=0.1, target_label=0):
    indices = np.random.choice(len(dataset), int(backdoor_rate * len(dataset)), replace=False)
    poisoned = []
    for i in range(len(dataset)):
        x, y = dataset[i]
        if i in indices:
            x_bd = x.clone()
            x_bd[:, :5, :5] = 3.0
            poisoned.append((x_bd, target_label))
        else:
            poisoned.append((x, y))
    return poisoned, indices

## Defense Suite

In [11]:
# TODO: Implement Defense Ablation study class
# HINT: Train models with each defense disabled and measure accuracy
class DefenseAblation:
    """Test each defense component independently."""
    def __init__(self, baseline_model, secure_model, test_loader):
        self.baseline = baseline_model
        self.secure = secure_model
        self.loader = test_loader
    
    def eval_no_dp(self):
        """Adversarial training only (no DP-SGD)."""
        # TODO: Create new model, train with epsilon_adv=0.1 but noise_multiplier=0
        # TODO: Evaluate on test set
        pass
    
    def eval_no_adversarial(self):
        """DP-SGD only (no adversarial training)."""
        # TODO: Create new model, train standard SGD on sanitized data
        # TODO: Evaluate on test set
        pass
    
    def eval_no_sanitization(self):
        """Adversarial + DP-SGD but on unsanitized data."""
        # TODO: Create new model, train with full config but on poisoned data
        # TODO: Evaluate on test set
        pass
    
    def run_ablation(self):
        """Run full ablation study."""
        # TODO: Return dict with:
        #   'Secure (All Defenses)': eval(self.secure)
        #   'DP-SGD Only': self.eval_no_dp()
        #   'Adversarial Only': self.eval_no_adversarial()
        #   'No Sanitization': self.eval_no_sanitization()
        #   'Baseline (No Defense)': eval(self.baseline)
        pass

## Evaluation Dashboard

In [12]:
# TODO: Create comprehensive security metrics dashboard
# HINT: Create DataFrame with 10+ metrics (accuracy, AUC, robustness, etc.)
# TODO: Create 6-panel visualization:
#   1. Accuracy comparison (baseline vs secure, clean vs adversarial)
#   2. Privacy metrics (MIA-AUC baseline vs secure)
#   3. Trojan detection (imbalance ratio)
#   4. Robustness-accuracy trade-off (epsilon sweep)
#   5. Defense ablation heatmap
#   6. Security scorecard
print('TODO: Implement evaluation dashboard visualization')

TODO: Implement evaluation dashboard visualization


## Exercises (10 Tasks)

**Part A: Attack Intensification (3 pts)**

### Exercise 1: Attack Scaling (1 pt)
TODO: Increase FGSM attack strength (ε) from 0.1 to 0.3 in steps of 0.05.  
Plot secure model accuracy vs ε. What ε causes accuracy to drop below 80%?

### Exercise 2: PGD Iterations (1 pt)
TODO: Vary PGD attack steps (5, 10, 20, 50) while keeping ε=0.1.  
Report accuracy for each step count. Does accuracy continue to drop beyond 20 steps?

### Exercise 3: Multi-Attack Ensemble (1 pt)
TODO: Combine FGSM + PGD + Sponge attacks simultaneously.  
What % of test samples are rejected/misclassified under combined attack?

---

**Part B: Poisoning & Trojans (3 pts)**

### Exercise 4: Poisoning Budget Trade-off (1 pt)
TODO: Vary poison rate (1%, 5%, 10%, 20%, 30%) on label-flip and backdoor attacks.  
For each rate, measure clean accuracy and target misclassification rate. Find the poisoning threshold.

### Exercise 5: Trojan Triggers (1 pt)
TODO: Create trojans with different trigger patterns (corner patch, center pixel, noise pattern).  
Which trigger is most stealthy (maintains clean accuracy)? Which is most effective (highest ASR)?

### Exercise 6: Trojan Detection Sensitivity (1 pt)
TODO: Adjust the trojan detection threshold (imbalance_ratio cutoff from 1.0 to 2.0).  
Measure detection True Positive Rate vs False Positive Rate. Plot an ROC curve.

---

**Part C: Privacy & Defenses (2 pts)**

### Exercise 7: Privacy-Utility Trade-off (1 pt)
TODO: Sweep DP noise multiplier (0.5, 1.0, 1.5, 2.0) and measure:
- Test accuracy (utility)
- Membership inference AUC (privacy)
Create a scatter plot of accuracy vs MIA-AUC. What's the optimal ε_dp for your threat model?

### Exercise 8: Defense Ablation (1 pt)
TODO: Disable each defense component and measure accuracy + robustness:
- Only DP-SGD (no adversarial training)
- Only adversarial training (no DP-SGD)  
- Only input sanitization (no adversarial training)  
Rank defenses by impact on PGD robustness. Which is most important?

---

**Part D: Advanced Topics (2 pts)**

### Exercise 9: Model Inversion on Defended Models (1 pt)
TODO: Run model inversion on both baseline and secure model.  
Compare the quality of inverted samples. Is the DP-SGD-trained model more resistant to inversion?  
*Hint: Measure L2 distance between inverted and random samples. Smaller = higher quality reconstruction = more privacy risk.*

### Exercise 10: Synthetic Data Utility (1 pt)
TODO: Train a classifier on 50% synthetic (VAE) + 50% real data. Evaluate on real test data.  
Does synthetic data help or hurt accuracy? Measure membership inference attack AUC on the hybrid-trained model.

---

**Bonus: Deployment Security Playbook**

Write a brief (5–10 bullet points) operational security checklist for deploying this secure ML system to production.  
Consider: monitoring, logging, incident response, model updates, adversarial monitoring.

## Deployment Checklist: Production Security Operationalization

In [13]:
# TODO: Create deployment security checklist
# Should include items for:
# - Pre-deployment (5 items): model audit, privacy audit, trojan scanning, adversarial testing, synthetic data QA
# - Runtime operations (5 items): input validation, rate limiting, logging, versioning, latency monitoring
# - Detection & response (5 items): anomaly detection, trojan detection, poisoning detection, incident response, notifications
# - Governance (5 items): GDPR compliance, audit trail, model cards, threat model updates, training
# - Continuous improvement (5 items): retraining, ablation studies, benchmarking, privacy audits, red teams

deployment_checklist = {
    # TODO: Add checklist items here
}

print('TODO: Implement deployment security checklist')

TODO: Implement deployment security checklist
